If the ligand signal is not recoverable in the data, we cannot expect scLEMBAS to learn perturbation separation. Rather, let's subset to the stronger perturbations in the dataset. 

Here, we rank order perturbations. Next, we select only those that are significantly separated from control.

In [42]:
import os

from tqdm import trange

import scanpy as sc
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns
from scipy.spatial.distance import cdist

import scperturb as scp

import sys
sclembas_path = '/home/hmbaghda/Projects/scLEMBAS'
sys.path.insert(1, os.path.join(sclembas_path))
from scLEMBAS import io
from scLEMBAS import latent_separation as ls

In [2]:
seed = 888
data_path = '/home/hmbaghda/orcd/pool/scLEMBAS/analysis'
author = 'McCauley'

n_cores = 30
os.environ["OMP_NUM_THREADS"] = '1'
os.environ["MKL_NUM_THREADS"] = '1'
os.environ["OPENBLAS_NUM_THREADS"] = '1'
os.environ["VECLIB_MAXIMUM_THREADS"] = '1'
os.environ["NUMEXPR_NUM_THREADS"] = '1'

In [3]:
tf_adata = io.read_tfad(os.path.join(data_path, 'interim', author + '_tf_activity_all.h5ad'))


/orcd/pool/005/hmbaghda/miniforge3/envs/scLEMBAS/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
cat_col = 'cell_type'
pert_col = 'ligand'
ctrl_pert = 'CTRL'

latent_label = 'pca'


In [5]:
# filter PCA space to number of components (since scperturb edist just takes them all)
tf_adata_sub = tf_adata.copy()
n_components = tf_adata_sub.uns['pca']['pca_rank']
tf_adata_sub.obsm['X_pca'] = tf_adata_sub.obsm['X_pca'][:, :n_components]

In [6]:
def bootstrap_median_ci(x, n_boot=5000, alpha=0.05, seed=0):
    rng = np.random.default_rng(seed)
    x = np.asarray(x)
    boots = np.array([np.median(rng.choice(x, size=len(x), replace=True))
                      for _ in range(n_boot)])
#     lo, hi = np.quantile(boots, [alpha/2, 1-alpha/2])
    hi = np.quantile(boots, 1 - alpha) # one-sided
    return hi


def bootstrap_delta_ci(D, p, control, n_boot=5000, alpha=0.05, seed=0):
    rng = np.random.default_rng(seed)

    vp = D.loc[p].drop(p).values
    vc = D.loc[control].drop(control).values

    n = min(len(vp), len(vc))  # or resample independently

    deltas = []
    for _ in range(n_boot):
        mp = np.median(rng.choice(vp, size=n, replace=True))
        mc = np.median(rng.choice(vc, size=n, replace=True))
        deltas.append(mp - mc)

    lo, hi = np.quantile(deltas, [alpha/2, 1 - alpha/2])
    return lo, hi


In [7]:
pert_strength_categorization = 'median_difference' # 'ctrl_upper_bound' 
pert_strength_alpha = 0.05 # 95% CI 

In [8]:
# get pairwise E-distances
dist = scp.edist(
    adata = tf_adata_sub, 
    obs_key = pert_col, 
    obsm_key = 'X_pca', 
    pwd = None, 
    dist = 'sqeuclidean', 
    sample_correct=True, 
    n_jobs = n_cores, 
    verbose = False
)

# aggregate by median across perturbations 
pert_strength = {}
for p in dist.index:
    row = dist.loc[p, :].drop(p)   # exclude self
    pert_strength[p] = row.median()
    
pert_strength = pd.DataFrame(data = {'perturbation': pert_strength.keys(), 'global_sep': pert_strength.values()})
pert_strength = pert_strength.sort_values(by = 'global_sep', ascending = False).reset_index(drop = True)


# compare pert to control by CI
if pert_strength_categorization == 'ctrl_upper_bound': # filter for those median perturbations > upper bound of ctrl CI
    # get the 95% confidence interval of the median estimate for the control condition
    ctrl_vec = dist.loc[ctrl_pert].drop(ctrl_pert).values
    ctrl_hi = bootstrap_median_ci(ctrl_vec, n_boot=5000, 
                                           alpha=pert_strength_alpha, seed=seed)
    pert_strength['strong'] = pert_strength.global_sep > ctrl_hi
elif pert_strength_categorization == 'median_difference':  # filter for those median perturbations differences with ctrl with lower CI > 0
    pert_strength_map = {}
    for pert in pert_strength.perturbation:
        if pert == ctrl_pert:
            pert_strength_map[pert] = False
            continue
        lo, hi = bootstrap_delta_ci(D = dist, p = pert, control = ctrl_pert, n_boot=5000, 
                                    alpha=pert_strength_alpha, seed=seed)
        pert_strength_map[pert] = lo > 0
    pert_strength['strong'] = pert_strength.perturbation.map(pert_strength_map)

    
pert_strength.to_csv(os.path.join(data_path, 'processed', author + '_pert_strength_quantification.csv'))
pert_strength


,perturbation,global_sep,strong
0,TGFB1,172.008469,True
1,IFNG,112.759664,True
2,IFNA2,80.925503,True
3,IL13,41.889279,True
4,BMP4,41.280189,True
5,IL17A,34.665664,False
6,OSM,34.049150,False
7,FGF10,29.411042,False
8,CHIR99021,27.376018,False
9,TNFA,20.737187,False


Re-run embedding:

In [44]:
covariates = [cat_col, pert_col, 'assigned_donor', 'phase']
comparison_metric = 'chance_adjusted_accuracy'

# same PLS arguments as in 01
csw = {
    'max_components': 25 ,
    'metric': 'accuracy', 
    'method': 'elbow', 
    'n_folds': 5, 
    'seed': 888
}

assessment_kwargs = {
    'n_perm': 100, 
    'get_q2_pval': True, 
    'get_r2_pval': False, 
    'get_accuracy_pval': False,
    'n_folds': 5, 
    'seed': 888
}

PCA quantification:

AxisArrays with keys: X_pls, X_umap, X_umap_pls

In [ ]:
io.write_tfad(tf_adata, 
              os.path.join(data_path, 'interim', author + '_tf_activity.h5ad')
             )